In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
from collections import deque

In [14]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

sp500 = pd.read_html(html)[0]

sp500.head()

tickers = sp500["Symbol"].tolist()

len(tickers)

tickers = [t.replace(".", "-") for t in tickers]

/var/folders/dw/zbs8s77s2_31t5fdxpgjcdjw0000gn/T/ipykernel_51686/3710868846.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500 = pd.read_html(html)[0]


In [15]:
prices = yf.download(
    tickers,
    start="2015-01-01",
    end="2025-12-31",
    auto_adjust=True,
    progress=True
)

[*********************100%***********************]  503 of 503 completed

1 Failed download:
['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2025-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1420088400, endDate = 1767157200")')


In [26]:
# 1) 日频收盘价
daily_close = prices["Close"].dropna(axis=1, how="all")

# 2) 月末收盘价
monthly_close = daily_close.resample("ME").last()

# 3) 月收益率
monthly_ret = monthly_close.pct_change(fill_method=None)

# # 4) 检查
# print(monthly_close.index[:5])
# print(monthly_ret.index[:5])
# print(monthly_close.shape)
# print(monthly_ret.shape)

In [28]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =========================
# 1) Data prep
# =========================
daily_close = prices["Close"].copy()
daily_close = daily_close.sort_index()
daily_close = daily_close.dropna(axis=1, how="all")   # 去掉全空 ticker

monthly_close = daily_close.resample("ME").last()
monthly_close = monthly_close.dropna(axis=1, how="all")

# 月收益率：用 fill_method=None，避免前向填充导致的数据污染
monthly_ret = monthly_close.pct_change(fill_method=None)

print("monthly_close index head:", monthly_close.index[:5])
print("monthly_ret index head:", monthly_ret.index[:5])
print("monthly_close shape:", monthly_close.shape)
print("monthly_ret shape:", monthly_ret.shape)


# =========================
# 2) Signal and deciles
# =========================
def compute_formation_signal(monthly_close: pd.DataFrame, J: int = 6) -> pd.DataFrame:
    """
    过去 J 个月累计收益率，用于形成期排序。
    signal[t] = P_t / P_{t-J} - 1
    """
    signal = monthly_close.pct_change(periods=J, fill_method=None)
    return signal


def assign_deciles(signal_row: pd.Series, n_deciles: int = 10) -> pd.Series:
    """
    单个月截面分组：
    1 = 最差分位（losers）
    10 = 最好分位（winners）
    """
    x = signal_row.dropna()
    if len(x) < n_deciles:
        return pd.Series(dtype="int64")

    # 用 first rank 避免 qcut 因为重复值报错
    ranks = x.rank(method="first")
    labels = pd.qcut(ranks, n_deciles, labels=False) + 1
    return pd.Series(labels.astype(int), index=x.index)


# =========================
# 3) Backtest
# =========================
def momentum_backtest(
    monthly_close: pd.DataFrame,
    monthly_ret: pd.DataFrame,
    J: int = 6,
    K: int = 6,
    n_deciles: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Jegadeesh-Titman style momentum backtest on monthly data.

    Rules:
    - 每个月末形成组合
    - 用过去 J 个月累计收益率排序
    - 买入最高分位（winners）
    - 卖出最低分位（losers）
    - 每个形成组合持有 K 个月
    - 同一月份中，所有仍在持有期内的 sub-portfolios 等权平均（overlapping portfolios）

    Returns
    -------
    strategy_df:
        每个月的组合收益、winner/loser 收益、活跃子组合数量
    holdings_df:
        每个 formation month 对应的赢家/输家名单
    """
    signal = compute_formation_signal(monthly_close, J=J)

    dates = monthly_close.index
    # 每个子组合记录：formation_date, start_pos, end_pos, winners, losers
    subportfolios = []

    holdings_rows = []
    for i, dt in enumerate(dates):
        if dt not in signal.index:
            continue

        row = signal.loc[dt]
        deciles = assign_deciles(row, n_deciles=n_deciles)
        if deciles.empty:
            continue

        winners = deciles[deciles == n_deciles].index.tolist()
        losers = deciles[deciles == 1].index.tolist()

        # 形成组合：在 dt 这一天末形成，下一期开始持有
        start_pos = i + 1
        end_pos = min(i + K, len(dates) - 1)  # 持有 K 个月：start_pos 到 start_pos+K-1

        if start_pos >= len(dates):
            continue

        subportfolios.append(
            {
                "formation_date": dt,
                "start_pos": start_pos,
                "end_pos": end_pos,
                "winners": winners,
                "losers": losers,
            }
        )

        holdings_rows.append(
            {
                "formation_date": dt,
                "n_winners": len(winners),
                "n_losers": len(losers),
                "winners": winners,
                "losers": losers,
            }
        )

    # 每个月计算所有活跃子组合的平均收益
    out = []
    for t_pos, dt in enumerate(dates):
        active_winner_rets = []
        active_loser_rets = []

        for sp in subportfolios:
            if sp["start_pos"] <= t_pos <= sp["end_pos"]:
                r = monthly_ret.loc[dt]

                w = sp["winners"]
                l = sp["losers"]

                wret = r[w].dropna().mean() if len(w) > 0 else np.nan
                lret = r[l].dropna().mean() if len(l) > 0 else np.nan

                if pd.notna(wret) and pd.notna(lret):
                    active_winner_rets.append(wret)
                    active_loser_rets.append(lret)

        if len(active_winner_rets) == 0:
            continue

        winner_ret = float(np.mean(active_winner_rets))
        loser_ret = float(np.mean(active_loser_rets))
        ls_ret = winner_ret - loser_ret

        out.append(
            {
                "date": dt,
                "winner_ret": winner_ret,
                "loser_ret": loser_ret,
                "long_short_ret": ls_ret,
                "n_active_subportfolios": len(active_winner_rets),
            }
        )

    strategy_df = pd.DataFrame(out).set_index("date")
    holdings_df = pd.DataFrame(holdings_rows).set_index("formation_date")
    return strategy_df, holdings_df


# =========================
# 4) Performance summary
# =========================
def performance_summary(r: pd.Series, periods_per_year: int = 12, nw_lags: int = 5) -> pd.Series:
    """
    计算均值、年化收益、波动率、Sharpe、Newey-West t-stat
    """
    r = r.dropna()
    if len(r) == 0:
        return pd.Series(dtype=float)

    mean_m = r.mean()
    vol_m = r.std(ddof=1)

    ann_ret = (1 + mean_m) ** periods_per_year - 1
    ann_vol = vol_m * np.sqrt(periods_per_year)
    sharpe = (mean_m / vol_m) * np.sqrt(periods_per_year) if vol_m > 0 else np.nan

    X = np.ones((len(r), 1))
    nw = sm.OLS(r.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})

    return pd.Series(
        {
            "mean_monthly": mean_m,
            "ann_return_approx": ann_ret,
            "ann_vol": ann_vol,
            "sharpe": sharpe,
            "nw_tstat_mean": nw.tvalues[0],
            "nw_pvalue_mean": nw.pvalues[0],
            "n_obs": len(r),
        }
    )


# =========================
# 5) Run the 6/6 strategy
# =========================
bt_66, holdings_66 = momentum_backtest(
    monthly_close=monthly_close,
    monthly_ret=monthly_ret,
    J=6,
    K=6,
    n_deciles=10,
)

print(bt_66.head())
print()
print(performance_summary(bt_66["long_short_ret"]))

# 可选：看看 winners / losers 的平均月收益
print()
print(bt_66[["winner_ret", "loser_ret", "long_short_ret"]].mean())

# 可选：累计净值曲线
equity_curve = (1 + bt_66["long_short_ret"].fillna(0)).cumprod()
print()
print(equity_curve.tail())

monthly_close index head: DatetimeIndex(['2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30',
               '2015-05-31'],
              dtype='datetime64[ns]', name='Date', freq='ME')
monthly_ret index head: DatetimeIndex(['2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30',
               '2015-05-31'],
              dtype='datetime64[ns]', name='Date', freq='ME')
monthly_close shape: (132, 502)
monthly_ret shape: (132, 502)
            winner_ret  loser_ret  long_short_ret  n_active_subportfolios
date                                                                     
2015-08-31   -0.050526  -0.064526        0.014000                       1
2015-09-30   -0.023500  -0.052382        0.028881                       2
2015-10-31    0.053361   0.107357       -0.053996                       3
2015-11-30    0.019164  -0.005910        0.025074                       4
2015-12-31   -0.013740  -0.054686        0.040946                       5

mean_monthly           0.003086
ann_return_

In [29]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =========================
# 0) Prepare monthly data
# =========================
daily_close = prices["Close"].copy()
daily_close = daily_close.sort_index()
daily_close = daily_close.dropna(axis=1, how="all")

monthly_close = daily_close.resample("ME").last()
monthly_close = monthly_close.dropna(axis=1, how="all")

monthly_ret = monthly_close.pct_change(fill_method=None)

print("monthly_close shape:", monthly_close.shape)
print("monthly_ret shape:", monthly_ret.shape)
print("monthly_close head index:", monthly_close.index[:5])
print("monthly_ret head index:", monthly_ret.index[:5])


# =========================
# 1) Formation signal
# =========================
def compute_formation_signal(monthly_close: pd.DataFrame, J: int) -> pd.DataFrame:
    """
    过去 J 个月累计收益率：
    signal[t] = P_t / P_{t-J} - 1
    """
    return monthly_close.pct_change(periods=J, fill_method=None)


def assign_deciles(signal_row: pd.Series, n_deciles: int = 10) -> pd.Series:
    """
    1 = losers decile
    10 = winners decile
    """
    x = signal_row.dropna()
    if len(x) < n_deciles:
        return pd.Series(dtype="int64")

    # ties 处理：先做 rank，再 qcut
    ranks = x.rank(method="first")
    labels = pd.qcut(ranks, n_deciles, labels=False) + 1
    return pd.Series(labels.astype(int), index=x.index)


# =========================
# 2) Strategy backtest
# =========================
def momentum_backtest(
    monthly_close: pd.DataFrame,
    monthly_ret: pd.DataFrame,
    J: int = 6,
    K: int = 6,
    n_deciles: int = 10,
    skip_months: int = 0,   # 月频数据里默认不跳月；留接口方便以后扩展
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Monthly momentum backtest:
    - 每个月末用过去 J 个月累计收益排序
    - 买入最高分位 winners
    - 卖出最低分位 losers
    - 持有 K 个月
    - 同一时点对所有活跃子组合等权平均（overlapping portfolios）
    """
    signal = compute_formation_signal(monthly_close, J=J)
    dates = monthly_close.index

    subportfolios = []
    holdings_rows = []

    for i, dt in enumerate(dates):
        if dt not in signal.index:
            continue

        row = signal.loc[dt]
        deciles = assign_deciles(row, n_deciles=n_deciles)
        if deciles.empty:
            continue

        winners = deciles[deciles == n_deciles].index.tolist()
        losers = deciles[deciles == 1].index.tolist()

        # 形成期结束后，经过 skip_months 再开始持有
        start_pos = i + 1 + skip_months
        end_pos = start_pos + K - 1

        if start_pos >= len(dates):
            continue
        end_pos = min(end_pos, len(dates) - 1)

        subportfolios.append(
            {
                "formation_date": dt,
                "start_pos": start_pos,
                "end_pos": end_pos,
                "winners": winners,
                "losers": losers,
            }
        )

        holdings_rows.append(
            {
                "formation_date": dt,
                "n_winners": len(winners),
                "n_losers": len(losers),
                "winners": winners,
                "losers": losers,
            }
        )

    out = []
    for t_pos, dt in enumerate(dates):
        active_winner_rets = []
        active_loser_rets = []

        for sp in subportfolios:
            if sp["start_pos"] <= t_pos <= sp["end_pos"]:
                r = monthly_ret.loc[dt]

                w = sp["winners"]
                l = sp["losers"]

                wret = r[w].dropna().mean() if len(w) > 0 else np.nan
                lret = r[l].dropna().mean() if len(l) > 0 else np.nan

                if pd.notna(wret) and pd.notna(lret):
                    active_winner_rets.append(wret)
                    active_loser_rets.append(lret)

        if len(active_winner_rets) == 0:
            continue

        winner_ret = float(np.mean(active_winner_rets))
        loser_ret = float(np.mean(active_loser_rets))
        ls_ret = winner_ret - loser_ret

        out.append(
            {
                "date": dt,
                "winner_ret": winner_ret,
                "loser_ret": loser_ret,
                "long_short_ret": ls_ret,
                "n_active_subportfolios": len(active_winner_rets),
            }
        )

    strategy_df = pd.DataFrame(out).set_index("date")
    holdings_df = pd.DataFrame(holdings_rows).set_index("formation_date")
    return strategy_df, holdings_df


# =========================
# 3) Performance summary
# =========================
def performance_summary(r: pd.Series, periods_per_year: int = 12, nw_lags: int = 5) -> pd.Series:
    """
    Monthly return series summary:
    - mean monthly return
    - approximate annual return
    - annual vol
    - Sharpe
    - Newey-West t-stat on mean
    """
    r = r.dropna()
    if len(r) == 0:
        return pd.Series(dtype=float)

    mean_m = r.mean()
    vol_m = r.std(ddof=1)

    ann_ret = (1 + mean_m) ** periods_per_year - 1
    ann_vol = vol_m * np.sqrt(periods_per_year)
    sharpe = (mean_m / vol_m) * np.sqrt(periods_per_year) if vol_m > 0 else np.nan

    X = np.ones((len(r), 1))
    nw = sm.OLS(r.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})

    return pd.Series(
        {
            "mean_monthly": mean_m,
            "ann_return_approx": ann_ret,
            "ann_vol": ann_vol,
            "sharpe": sharpe,
            "nw_tstat_mean": nw.tvalues[0],
            "nw_pvalue_mean": nw.pvalues[0],
            "n_obs": len(r),
        }
    )


# =========================
# 4) Run all J/K combos
# =========================
J_list = [3, 6, 9, 12]
K_list = [3, 6, 9, 12]

all_results = {}
summary_rows = []

for J in J_list:
    for K in K_list:
        bt, holdings = momentum_backtest(
            monthly_close=monthly_close,
            monthly_ret=monthly_ret,
            J=J,
            K=K,
            n_deciles=10,
            skip_months=0,
        )

        all_results[(J, K)] = {
            "bt": bt,
            "holdings": holdings,
        }

        summ = performance_summary(bt["long_short_ret"])
        summ["J"] = J
        summ["K"] = K
        summ["n_obs_bt"] = len(bt)
        summ["mean_winner"] = bt["winner_ret"].mean()
        summ["mean_loser"] = bt["loser_ret"].mean()
        summ["mean_ls"] = bt["long_short_ret"].mean()
        summary_rows.append(summ)

summary_df = pd.DataFrame(summary_rows).set_index(["J", "K"]).sort_index()

print(summary_df)

# 方便看一眼最关键的列
display(
    summary_df[[
        "mean_winner",
        "mean_loser",
        "mean_ls",
        "nw_tstat_mean",
        "sharpe",
        "n_obs",
    ]].round(4)
)


# =========================
# 5) Optional: pivot table / heatmap ready
# =========================
ls_mean_table = summary_df["mean_ls"].unstack("K")
tstat_table = summary_df["nw_tstat_mean"].unstack("K")

print("\nMean long-short return table:")
display(ls_mean_table.round(4))

print("\nNW t-stat table:")
display(tstat_table.round(2))


# =========================
# 6) Look at one strategy in detail
# =========================
bt_66 = all_results[(6, 6)]["bt"]
holdings_66 = all_results[(6, 6)]["holdings"]

print("\n6/6 strategy head:")
display(bt_66.head())

print("\n6/6 strategy summary:")
display(performance_summary(bt_66["long_short_ret"]).round(4))

print("\n6/6 holdings head:")
display(holdings_66.head())


monthly_close shape: (132, 502)
monthly_ret shape: (132, 502)
monthly_close head index: DatetimeIndex(['2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30',
               '2015-05-31'],
              dtype='datetime64[ns]', name='Date', freq='ME')
monthly_ret head index: DatetimeIndex(['2015-01-31', '2015-02-28', '2015-03-31', '2015-04-30',
               '2015-05-31'],
              dtype='datetime64[ns]', name='Date', freq='ME')
           mean_monthly  ann_return_approx   ann_vol    sharpe  nw_tstat_mean  \
J    K                                                                          
3.0  3.0       0.000704           0.008478  0.160449  0.052634       0.222538   
     6.0       0.002107           0.025578  0.116677  0.216696       1.041935   
     9.0       0.001384           0.016740  0.107615  0.154370       0.639229   
     12.0      0.001052           0.012695  0.095099  0.132723       0.487089   
6.0  3.0       0.003615           0.044253  0.171094  0.253544       1.24325

mean_winner  mean_loser  mean_ls  nw_tstat_mean  sharpe  n_obs
J    K                                                                   
3.0  3.0        0.0178      0.0171   0.0007         0.2225  0.0526  128.0
     6.0        0.0181      0.0160   0.0021         1.0419  0.2167  128.0
     9.0        0.0179      0.0165   0.0014         0.6392  0.1544  128.0
     12.0       0.0177      0.0166   0.0011         0.4871  0.1327  128.0
6.0  3.0        0.0202      0.0166   0.0036         1.2433  0.2535  125.0
     6.0        0.0200      0.0169   0.0031         1.0196  0.2354  125.0
     9.0        0.0195      0.0174   0.0021         0.6254  0.1722  125.0
     12.0       0.0191      0.0176   0.0015         0.4588  0.1344  125.0
9.0  3.0        0.0207      0.0174   0.0033         0.8460  0.2061  122.0
     6.0        0.0200      0.0176   0.0025         0.5915  0.1682  122.0
     9.0        0.0198      0.0180   0.0018         0.4211  0.1295  122.0
     12.0       0.0196      0.0183   0.0013         0.3155  0.0966  122.0
12.0 3.0        0.0208      0.0199   0.0009         0.1961  0.0565  119.0
     6.0        0.0213      0.0203   0.0011         0.2188  0.0671  119.0
     9.0        0.0211      0.0205   0.0006         0.1189  0.0368  119.0
     12.0       0.0206      0.0205   0.0001         0.0174  0.0054  119.0


Mean long-short return table:


K,3.0,6.0,9.0,12.0
J,,,,
3.0,0.0007,0.0021,0.0014,0.0011
6.0,0.0036,0.0031,0.0021,0.0015
9.0,0.0033,0.0025,0.0018,0.0013
12.0,0.0009,0.0011,0.0006,0.0001



NW t-stat table:


...

,winner_ret,loser_ret,long_short_ret,n_active_subportfolios
date,,,,
2015-08-31,-0.050526,-0.064526,0.014000,1
2015-09-30,-0.023500,-0.052382,0.028881,2
2015-10-31,0.053361,0.107357,-0.053996,3
2015-11-30,0.019164,-0.005910,0.025074,4
2015-12-31,-0.013740,-0.054686,0.040946,5



6/6 strategy summary:


mean_monthly           0.0031
ann_return_approx      0.0377
ann_vol                0.1573
sharpe                 0.2354
nw_tstat_mean          1.0196
nw_pvalue_mean         0.3079
n_obs                125.0000
dtype: float64


6/6 holdings head:


,n_winners,n_losers,winners,losers
formation_date,,,,
2015-07-31,46,47,"[AIG, AKAM, AMZN, ANET, BLDR, CI, CIEN, CNC, C...","[AMAT, AMD, APA, APO, BEN, BIIB, CNP, COP, CVX..."
2015-08-31,46,47,"[ACGL, AIZ, AMZN, BLDR, CAG, CASY, CCL, CI, DX...","[ADSK, ALB, AMAT, AMD, APA, BEN, BIIB, COP, CS..."
2015-09-30,47,47,"[ACGL, ADBE, AIZ, AMZN, BLDR, CAG, CASY, CBOE,...","[ADSK, AES, AMAT, AMD, APA, BEN, BIIB, CNC, CV..."
2015-10-31,47,47,"[ACGL, AIZ, AMZN, AOS, CASY, CCL, CINF, DXCM, ...","[AKAM, APA, APO, AXON, BEN, BIIB, BKR, CMI, CO..."
2015-11-30,47,47,"[AIZ, AMZN, CASY, CBOE, CINF, DHI, DXCM, ERIE,...","[ADM, AES, AKAM, APO, ARES, AXON, BG, BIIB, BX..."


We can find that if we fix the lagged time, the return will genuly decrease as the holding period increases. And note when J = 12, nearly every K would not get a good result. The best strategy is (6,3) which differs from the paper which is fair as the market becomes more mature. A natural problem is: has the optimal formation period of momentum strategies shortened in modern equity markets?

In [16]:
import pyarrow
print(pyarrow.__version__)
from pathlib import Path

# 如果 data 文件夹不存在就创建
Path("../data").mkdir(exist_ok=True)

# 保存价格数据
prices.to_parquet("../data/sp500_prices.parquet")

print("Saved.")

24.0.0
Saved.


In [11]:
import sys

!{sys.executable} -m pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 4.4 MB/s  0:00:08m0:00:0100:01

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
